# 08 — Final Acceptance Checks

Run after Notebook 07.

A real project is complete only when correctness checks pass.


In [0]:
import sys
import os
from pathlib import Path

marker = "RETAIL_LAB.json"
cwd = Path(os.getcwd()).resolve()
PROJECT_ROOT = None

for candidate in [cwd, *cwd.parents]:
    if (candidate / marker).exists():
        PROJECT_ROOT = str(candidate)
        break

if PROJECT_ROOT is None:
    for search_root in [Path("/Workspace/Users"), Path("/Workspace/Shared"), Path("/Workspace")]:
        if not search_root.exists():
            continue
        for current_root, dirs, files in os.walk(search_root):
            dirs[:] = [d for d in dirs if not d.startswith(".") and d not in {"Trash", ".Trash"}]
            if marker in files:
                PROJECT_ROOT = current_root
                break
        if PROJECT_ROOT:
            break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project not found. Import the complete ZIP into Databricks Workspace."
    )

SRC_ROOT = f"{PROJECT_ROOT}/src"
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

from .bootstrap import ensure_lab

ctx = ensure_lab(
    spark=spark,
    project_root=PROJECT_ROOT,
)

print("Project root :", PROJECT_ROOT)
print("Catalog      :", ctx.catalog)
print("Schema       :", ctx.schema)
print("Volume       :", ctx.volume)
print("Source root  :", ctx.source_root)


In [0]:
assert spark.table(ctx.table("silver_customers")).count() == 12
assert spark.table(ctx.table("silver_orders")).count() == 126
assert spark.table(ctx.table("gold_order_sales")).count() == 259

print("Managed-table counts: PASSED")


In [0]:
orders_df = spark.table(ctx.table("silver_orders"))

actual = {
    r["order_status"]: r["count"]
    for r in orders_df.groupBy("order_status").count().collect()
}

expected = {
    "DELIVERED": 78,
    "SHIPPED": 29,
    "PLACED": 10,
    "CANCELLED": 9,
}

print(actual)
assert actual == expected
print("Order status counts: PASSED")


In [0]:
from pyspark.sql import functions as F

skew_df = (
    spark.read.option("header", True)
    .csv(f"{ctx.source_root}/skew_sales.csv")
)

assert skew_df.filter(F.col("city") == "Hyderabad").count() == 700
print("Skew acceptance: PASSED")


In [0]:
display(
    spark.sql(
        f"SHOW TABLES IN `{ctx.catalog}`.`{ctx.schema}`"
    )
)

print("PASSED")
